# Metadados, identificadores e proveniência

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/andrelfo85/Atividade_Goodreads/blob/master/unidade_02/03_metadados_identificadores_e_proveniencia.ipynb)

Os notebooks anteriores definiram o corpus e analisaram seus limites. Este
notebook pergunta como preservar essas decisões junto aos dados para que
outra pessoa consiga compreender, verificar e reutilizar a base.

## 1. Documentação como parte da base

Metadados permitem compreender, localizar, relacionar, administrar e
reutilizar registros. Para esta unidade, distinguiremos:

- **descritivos:** título, autoria, data, assunto;
- **administrativos:** direitos, acesso, formato, responsável;
- **estruturais:** relações entre partes, páginas ou versões;
- **proveniência:** origem, agentes, datas e transformações.

As categorias se sobrepõem em padrões reais; servem aqui como guia de
inspeção.

Saber que tipos de metadados existem é apenas o primeiro nível. O
dicionário de dados traduz essa visão para cada campo, explicitando seu
significado, sua origem e suas regras.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_02'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /content/disciplina_computacao_aplicada_humanidades_digitais/unidade_02


## 2. Dicionário de dados

Um dicionário deve registrar ao menos nome, definição, tipo, valores ou
regras, origem e limitações. Ele documenta o significado esperado; não
garante que os registros estejam corretos.

In [2]:
import json
import pandas as pd

catalogo = pd.read_csv("dados/catalogo_fontes.csv")
dicionario = pd.read_csv("dados/dicionario_dados.csv")
dicionario

,campo,descricao,tipo,regra,origem
0,id_fonte,Identificador estável no catálogo didático,texto,único e não vazio,equipe docente
1,instituicao,Custodiante fictício da fonte,categoria,quatro valores previstos,catálogo simulado
2,tipo_documental,Gênero atribuído ao item,categoria,vocabulário controlado,catálogo simulado
3,ano,Ano atribuído ao item,inteiro,1890 a 1904,catálogo simulado
4,local,Recorte espacial amplo,categoria,Capital ou Interior,catálogo simulado
5,grupo_representado,Grupo mais diretamente registrado,categoria,interpretação didática,equipe docente
6,localizado,Item localizado no levantamento,categoria,sim ou não,levantamento simulado
7,digitalizado,Representação digital disponível,categoria,sim ou não,levantamento simulado
8,condicao_acesso,Condição informada pelo custodiante,categoria,não equivale a licença de uso,catálogo simulado
9,qualidade_metadados,Completude descritiva aproximada,ordinal,"mínimo, parcial ou completo",equipe docente


Definidos os campos, precisamos manter as relações entre registros mesmo
quando títulos, posições ou descrições mudam. Essa função é exercida pelos
identificadores.

## 3. Identificadores

Um identificador deve ser único no escopo definido, persistente o bastante
para manter relações e independente de atributos que podem mudar. Título
ou número da linha são candidatos frágeis. O teste abaixo verifica
unicidade e ausência, mas não prova persistência institucional.

In [3]:
auditoria_id = {
    "registros": len(catalogo),
    "ids_ausentes": int(catalogo["id_fonte"].isna().sum()),
    "ids_duplicados": int(catalogo["id_fonte"].duplicated().sum()),
    "ids_unicos": int(catalogo["id_fonte"].nunique()),
}
auditoria_id

{'registros': 16, 'ids_ausentes': 0, 'ids_duplicados': 0, 'ids_unicos': 16}

A unicidade dos identificadores garante uma condição estrutural, mas não
assegura que o restante da base respeite o esquema documentado. A próxima
auditoria compara campos e valores observados com o que era esperado.

## 4. Campos e domínios esperados

Esta auditoria compara o esquema observado com o documentado e verifica
domínios simples. Diagnosticar não é limpar: decisões de padronização serão
trabalhadas na Unidade 3.

In [4]:
campos_documentados = set(dicionario["campo"])
campos_observados = set(catalogo.columns)
print("Sem documentação:", sorted(campos_observados - campos_documentados))
print("Documentados e ausentes:", sorted(campos_documentados - campos_observados))

dominios_esperados = {
    "localizado": {"sim", "não"},
    "digitalizado": {"sim", "não"},
    "qualidade_metadados": {"mínimo", "parcial", "completo"},
}
for campo, dominio in dominios_esperados.items():
    inesperados = set(catalogo[campo].dropna()) - dominio
    print(campo, "valores inesperados:", inesperados)

Sem documentação: []
Documentados e ausentes: []
localizado valores inesperados: set()
digitalizado valores inesperados: set()
qualidade_metadados valores inesperados: set()


Essas verificações descrevem o estado atual da tabela. Para explicar como
ela chegou a esse estado — fontes, agentes, datas e transformações — é
necessário acrescentar proveniência.

## 5. Proveniência

No diagrama, o dicionário explica o significado dos campos, enquanto a
proveniência registra a trajetória entre fonte, atividade, agente e dado
derivado.

![Cadeia conecta fonte, registro e identificador, transformação e dado derivado; o dicionário de dados define os campos e a proveniência registra entidades, atividades, agentes, versões e datas.](https://github.com/andrelfo85/Atividade_Goodreads/blob/master/unidade_02/imagens/03_documentacao_proveniencia.svg?raw=1)

Proveniência responde: quem criou ou custodiou o registro, de onde ele
veio, quando foi obtido, sob quais condições e que transformações sofreu?
Ela forma uma cadeia entre fonte, representação e resultado. Um endereço
eletrônico sozinho não documenta data de acesso, versão, autoria nem
transformação.

In [36]:
with open("dados/proveniencia_catalogo.json", encoding="utf-8") as arquivo:
    proveniencia = json.load(arquivo)

for chave in ["titulo", "natureza", "criado_em", "responsavel", "origem"]:
    print(f"{chave}: {proveniencia[chave]}")
print("Transformações registradas:", len(proveniencia["transformacoes"]))

titulo: Catálogo didático de fontes
natureza: dados inteiramente fictícios
criado_em: 2026-07-30
responsavel: equipe docente
origem: registros simulados, sem correspondência com acervos reais
Transformações registradas: 3


Dicionário, identificadores, validações e proveniência cumprem funções
distintas. A atividade reúne essas camadas em uma documentação coerente
para o projeto do estudante.

## U02-A06 — Atividade — Documentação do Projeto Goodreads

As células de código seguintes aplicam ao projeto as mesmas verificações do
notebook (identificadores, campos documentados e proveniência).

**Estratégia de identificadores e escopo de unicidade:** Uso os
identificadores do próprio Goodreads. `id_livro` é único na tabela de
livros (verifiquei 10.454 valores únicos em 10.454 linhas). `id_autor` é
único na tabela de autores (7.780 autores) e funciona como chave estrangeira na
tabela de livros. O escopo de unicidade é o recorte recebido. Limites: o mesmo
nome de autor pode ter mais de um id (Por exemplo: William Gibson aparece com dois ids, um deles associado a *The Miracle Worker*, que provavelmente pertence a um homônimo); e o id do livro pode identificar uma edição específica, não a obra. Os ids são persistentes apenas enquanto o Goodreads os mantiver.

**Metadados mínimos e justificativa:** Título, autor e ano (descritivos, para
reencontrar o livro); `id_livro` e `id_autor` (identificação e ligação entre
tabelas); gêneros literários (contexto analítico); data e método de coleta,
responsável e termos de uso (administrativos e de proveniência, no momento ausentes); hash do arquivo (para provar que a cópia não mudou).

**Campos do dicionário, definições e domínios:** Documentados no arquivo
`projeto_goodreads/dados/dicionario_dados.csv`, com papel, descrição, tipo,
domínio ou regra, origem e limitação de cada um dos 15 campos. Exemplos:
`genero_autor` (categoria herdada; domínio feminino ou masculino; regra de
origem desconhecida); `ano_publicacao` (inteiro de −720 a 2019; negativo indica
antes de Cristo); `n_resenhas` (inteiro, nunca maior que `n_avaliacoes`).

**Origem, custodiante, versão e data de acesso:** Origem declarada: recorte
extraído do Goodreads, obtido para resolução das atividads da disciplina. Custodiante da cópia local: Eu, o pesquisador. Versão: identificada pelo hash SHA-256 `40effdd6…b9cfe64` e pelo tamanho (1.345.736 bytes). Data de obtenção: set-2026. Data de coleta no Goodreads: desconhecida (ano máximo de publicação 2019). Tudo registrado em
`projeto_goodreads/dados/brutos/proveniencia_goodreads.json`.

**Transformações previstas e responsável por registrá-las:** Padronização do
local de nascimento (preservando o original), sinalização de anos, páginas e
contagens incoerentes, cálculo da taxa de resenhas, separação em tabelas de
livros, autores e relações livro–gênero, auditoria do rótulo de gênero com
o nome. O responsável sou seu, o pesquisador, e cada regra fica registrada no código `preparar_base.py` e no log da Unidade 3.

**Relação entre registro derivado e fonte:** Cada linha derivada conserva
`id_livro` e `id_autor`, que permitem voltar à página do livro e do autor no
Goodreads; cada coluna transformada tem a coluna original ao lado; o arquivo
bruto é imutável e verificável pelo hash.

Revise o conjunto perguntando se uma pessoa externa conseguiria interpretar
os campos e reconstruir a trajetória de um registro. Essa verificação leva
à síntese e evidencia o que a documentação técnica ainda não resolve.

In [33]:
import subprocess
from pathlib import Path
import shutil

# URL do meu repositório específico contendo o projeto_goodreads
URL_PROJETO_USER = 'https://github.com/andrelfo85/Atividade_Goodreads.git'
PASTA_CLONADA = Path('/content/Atividade_Goodreads')
PASTA_DESTINO = Path('/content/projeto_goodreads')

if not PASTA_DESTINO.exists():
    print("Clonando repositório do projeto...")
    # Clona o repositório temporariamente
    subprocess.run(["git", "clone", "--depth", "1", URL_PROJETO_USER, str(PASTA_CLONADA)], check=True)

    # Move a pasta 'projeto_goodreads' para /content/projeto_goodreads
    origem_projeto = PASTA_CLONADA / "projeto_goodreads"
    if origem_projeto.exists():
        shutil.move(str(origem_projeto), str(PASTA_DESTINO))
        print("Pasta 'projeto_goodreads' configurada com sucesso em /content!")
    else:
        print("Erro: A pasta 'projeto_goodreads' não foi encontrada dentro do repositório clonado.")

    # Remove o restante dos arquivos clonados para manter o ambiente limpo
    if PASTA_CLONADA.exists():
        shutil.rmtree(PASTA_CLONADA)
else:
    print("A pasta 'projeto_goodreads' já está pronta para uso em /content.")

A pasta 'projeto_goodreads' já está pronta para uso em /content.


In [34]:
# Localiza a pasta do projeto Goodreads (localmente ou no Colab, se a pasta for enviada para /content)
import sys
import pandas as pd

CANDIDATOS = [Path("../projeto_goodreads"), Path("projeto_goodreads"), Path("/content/projeto_goodreads")]
PROJETO = next((c.resolve() for c in CANDIDATOS if (c / "preparar_base.py").exists()), None)
if PROJETO is None:
    raise FileNotFoundError(
        "Pasta projeto_goodreads não encontrada."
    )
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))
print("Projeto localizado em:", PROJETO)
#print("Arquivo preparar_base.py existe:", (PROJETO / "preparar_base.py").exists())

from preparar_base import ler_bruto, sha256_arquivo, BRUTO

livros_gr = ler_bruto()
dicionario_gr = pd.read_csv(PROJETO / "dados" / "dicionario_dados.csv")

auditoria_id_gr = {
    "Registros de livro": len(livros_gr),
    "id de livro ausentes": int(livros_gr["id_livro"].isna().sum()),
    "id de livro duplicados": int(livros_gr["id_livro"].duplicated().sum()),
    "id de autor distintos": int(livros_gr["id_autor"].nunique()),
    "Nomes de autor com mais de um id": int(livros_gr.groupby("autor")["id_autor"].nunique().gt(1).sum()),
}
print("\n",auditoria_id_gr)
print("\nSem documentação:", sorted(set(livros_gr.columns) - set(dicionario_gr["campo"])))
print("\nDocumentados e ausentes:", sorted(set(dicionario_gr["campo"]) - set(livros_gr.columns)))
print("\nDomínio de genero_autor:", sorted(livros_gr["genero_autor"].unique()))

Projeto localizado em: /content/projeto_goodreads

 {'Registros de livro': 10454, 'id de livro ausentes': 0, 'id de livro duplicados': 0, 'id de autor distintos': 7780, 'Nomes de autor com mais de um id': 1}

Sem documentação: []

Documentados e ausentes: []

Domínio de genero_autor: ['feminino', 'masculino']


In [35]:
with open(PROJETO / "dados" / "brutos" / "proveniencia_goodreads.json", encoding="utf-8") as arquivo:
    proveniencia_gr = json.load(arquivo)
print("Hash confere com a proveniência:", sha256_arquivo(BRUTO) == proveniencia_gr["sha256"])
for item in proveniencia_gr["origem_desconhecida"]:
    print("- desconhecido:", item)
dicionario_gr[["campo", "papel", "origem", "limitacao"]]

Hash confere com a proveniência: True
- desconhecido: quem realizou a extração original
- desconhecido: data exata da coleta (o ano de publicação mais recente é 2019, o que sugere coleta em 2019 ou depois)
- desconhecido: método de coleta (API, raspagem ou outro)
- desconhecido: critério de seleção dos 10.454 livros
- desconhecido: método de atribuição do campo genero_autor
- desconhecido: data e método de tradução dos nomes das colunas para o português


,campo,papel,origem,limitacao
0,id_livro,identificador,Goodreads via arquivo recebido,identifica a edição ou obra catalogada; não ga...
1,titulo,metadado descritivo,Goodreads,129 títulos sem letras latinas; marcadores de ...
2,id_autor,identificador,Goodreads,um autor por livro; coautorias ausentes; um me...
3,genero_1,variável categórica,estantes criadas por usuários do Goodreads (re...,"rótulos da comunidade, não classificação edito..."
4,genero_2,variável categórica,estantes de usuários do Goodreads,idem
5,paginas,variável numérica (razão),Goodreads,29 valores suspeitos (até 10 ou acima de 2000)...
6,ano_publicacao,variável temporal,Goodreads,não se sabe se é primeira publicação ou edição...
7,nota_livro,variável numérica,uso da rede,média sem distribuição; não mede qualidade lit...
8,n_avaliacoes,variável de reconhecimento,uso da rede,"valor no momento da coleta, data desconhecida"
9,n_resenhas,variável de reconhecimento,uso da rede,não informa o conteúdo das resenhas


## Leituras e síntese

O modelo PROV-O do W3C oferece conceitos para representar entidades,
atividades e agentes. Ele não substitui a descrição arquivística ou os
padrões específicos do campo; oferece uma linguagem para tornar relações
e decisões rastreáveis. No Notebook 04, a documentação será ampliada para
motivação, composição, usos, distribuição e manutenção da base a partir
da proposta de *Datasheets for Datasets*.

Uma base auditável combina documentação legível por pessoas com
verificações computacionais simples. Dados completos: `referencias.md`.

Leve essa documentação ao Notebook 04. Nele, a pergunta se amplia: mesmo
uma base tecnicamente auditável ainda precisa justificar acesso, reuso,
autoridade, benefício e responsabilidades.